# GAIE 2027 – run the LLM judges on a free Kaggle GPU

Runs every LLM judge in the protocol on the **frozen** test set and gives you a zip of `runs/` to copy into
`evaluation/data/test/runs/` on your laptop. Then run `python -m evaluation.report` there.

**Before you start**
1. *Settings → Accelerator → GPU T4 x2* and *Settings → Internet → On*.
2. Upload your test folder (`items.jsonl`, `responses.jsonl`, `excluded.csv`, `selected_items.txt`, `typing_guide.txt`,
   `typing_assignment.csv`, `typed_by_classmates.csv`, `FREEZE_specs.json`, `FREEZE.json`) as a
   private Kaggle **Dataset** called `igcse-test-set`, then *Add Input* → that dataset.
   Do not upload `gold.csv`: the judges never need the labels.
3. *Run all*. Every run resumes where it stopped, so if the session ends just run all again.

All judges run here with the same library (transformers, bf16/fp16), k = 5 samples at temperature 0.7 and seeds 0–4,
so they are compared fairly. Each run records the model's exact revision and the seeds in `runs/<judge>.meta.json`
(protocol §2: the laptop and Kaggle runs must provably use the same weights). Use this only under the agreed
fallback rule: if the laptop pilot shows the runs cannot finish by 1 December.

In [ ]:
REPO_URL = "https://github.com/wandiyajames-arch/igcse_math_tutor.git"   # or Angelah's repo (needs a token)
DATA_DIR = "/kaggle/input/igcse-test-set"       # the dataset you added
SPLIT = "test"                                   # use "dev" first to check timing and parsing
LIMIT = None                                     # e.g. 20 for a quick timing check

JUDGES = [  # (system name, backend, prompt kind)
    ("compass_3b", "hf:opencompass/CompassVerifier-3B", "compass"),
    ("qwen3_4b",   "hf:Qwen/Qwen3-4B",                  "examiner"),
    ("qwen25_7b",  "hf:Qwen/Qwen2.5-7B-Instruct",       "examiner"),
    # ("compass_7b", "hf:opencompass/CompassVerifier-7B", "compass"),   # optional, if time allows
]

In [ ]:
!git clone -q $REPO_URL /kaggle/working/repo || (cd /kaggle/working/repo && git pull -q)
%cd /kaggle/working/repo
!pip install -q -r requirements.txt -r requirements-eval.txt "transformers>=4.51" accelerate
!git log --oneline -1

In [ ]:
import shutil, pathlib, subprocess
dst = pathlib.Path(f"evaluation/data/{SPLIT}")
dst.mkdir(parents=True, exist_ok=True)
for f in pathlib.Path(DATA_DIR).iterdir():
    if f.is_file():
        shutil.copy(f, dst / f.name)
print(sorted(p.name for p in dst.iterdir()))
# The freeze must pass before any judge runs: data, grader and prompts unchanged since the freeze.
r = subprocess.run(["python", "-m", "evaluation.freeze", "verify", str(dst)], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert SPLIT == "dev" or r.returncode == 0, "freeze check failed – do not run the judges on changed data"


In [ ]:
import gc, subprocess, torch
for name, backend, kind in JUDGES:
    cmd = ["python", "-m", "evaluation.systems", str(dst), "--system", name, "--backend", backend, "--kind", kind, "--k", "5"]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    print(">>>", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=False)
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Rule-based systems are fast; run them here too so everything comes from one machine.
!python -m evaluation.systems {dst} --system ours
!python -m evaluation.systems {dst} --system math_verify
!python -m evaluation.systems {dst} --system string_match
!cd {dst} && zip -qr /kaggle/working/runs_{SPLIT}.zip runs && ls -la runs
print("Download /kaggle/working/runs_%s.zip from the Output panel (right side)." % SPLIT)